# 07 - Qualidade dos Dados

## Objetivo

Avaliar a qualidade das tabelas da camada Silver antes da construção
da camada Gold.

## Fontes avaliadas

- CETIC.br - TIC Domicílios 2025
- YouTube Trending Brasil 2025
- Google Trends 2025

## Dimensões de qualidade

- Completude
- Unicidade
- Validade
- Consistência
- Domínio
- Integridade
- Análise de possíveis outliers

Este notebook não altera os registros das tabelas Silver.
Problemas identificados serão documentados e, quando necessário,
corrigidos na etapa de transformação correspondente.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
df_cetic = spark.table(
    "workspace.mvp_silver.consumo_digital_cetic_2025"
)

df_youtube = spark.table(
    "workspace.mvp_silver.youtube_trending_br_2025"
)

df_trends = spark.table(
    "workspace.mvp_silver.google_trends_2025"
)

print("CETIC:", df_cetic.count())
print("YouTube:", df_youtube.count())
print("Google Trends:", df_trends.count())

In [0]:
campos_esperados = {
    "CETIC": [
        "id_respondente",
        "faixa_etaria_codigo",
        "faixa_etaria",
        "categoria",
        "resposta_codigo",
        "resposta_valida",
        "peso"
    ],

    "YouTube": [
        "video_id",
        "data_trending",
        "categoria_original",
        "categoria",
        "nivel_comparabilidade",
        "views",
        "likes",
        "comments"
    ],

    "Google Trends": [
        "data_semana",
        "categoria",
        "indice_trends"
    ]
}

dataframes = {
    "CETIC": df_cetic,
    "YouTube": df_youtube,
    "Google Trends": df_trends
}

for fonte, campos in campos_esperados.items():

    print(f"\n=== {fonte} ===")

    for campo in campos:

        print(
            campo,
            "->",
            "OK"
            if campo in dataframes[fonte].columns
            else "NÃO ENCONTRADO"
        )

In [0]:
display(
    df_cetic.agg(

        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("id_respondente").isNull(),
                1
            ).otherwise(0)
        ).alias("id_respondente_nulo"),

        F.sum(
            F.when(
                F.col("faixa_etaria").isNull(),
                1
            ).otherwise(0)
        ).alias("faixa_etaria_nula"),

        F.sum(
            F.when(
                F.col("categoria").isNull(),
                1
            ).otherwise(0)
        ).alias("categoria_nula"),

        F.sum(
            F.when(
                F.col("resposta_codigo").isNull(),
                1
            ).otherwise(0)
        ).alias("resposta_codigo_nula"),

        F.sum(
            F.when(
                F.col("peso").isNull(),
                1
            ).otherwise(0)
        ).alias("peso_nulo")
    )
)

In [0]:
duplicados_cetic = (
    df_cetic
    .groupBy(
        "id_respondente",
        "categoria"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

print(
    "Duplicidades id_respondente + categoria:",
    duplicados_cetic.count()
)

In [0]:
total_respondentes = (
    df_cetic
    .select("id_respondente")
    .distinct()
    .count()
)

total_categorias = (
    df_cetic
    .select("categoria")
    .distinct()
    .count()
)

total_esperado = (
    total_respondentes
    * total_categorias
)

total_real = df_cetic.count()

print("Respondentes:", total_respondentes)
print("Categorias:", total_categorias)
print("Registros esperados:", total_esperado)
print("Registros reais:", total_real)
print(
    "Estrutura consistente:",
    total_esperado == total_real
)

In [0]:
categorias_cetic_validas = [
    "Notícias",
    "Esportes",
    "Música",
    "Humor",
    "Animações",
    "Games",
    "Tutoriais / Educação",
    "Influenciadores"
]

display(
    df_cetic
    .groupBy("categoria")
    .count()
    .orderBy("categoria")
)

In [0]:
categorias_invalidas_cetic = (
    df_cetic
    .filter(
        ~F.col("categoria")
        .isin(categorias_cetic_validas)
    )
    .count()
)

print(
    "Categorias fora do domínio:",
    categorias_invalidas_cetic
)

In [0]:
faixas_validas = [
    "10-15",
    "16-24",
    "25-34",
    "35-44",
    "45-59",
    "60+"
]

display(
    df_cetic
    .groupBy(
        "faixa_etaria_codigo",
        "faixa_etaria"
    )
    .count()
    .orderBy("faixa_etaria_codigo")
)

In [0]:
faixas_invalidas = (
    df_cetic
    .filter(
        ~F.col("faixa_etaria")
        .isin(faixas_validas)
    )
    .count()
)

print(
    "Faixas etárias inválidas:",
    faixas_invalidas
)

In [0]:
display(
    df_cetic
    .groupBy(
        "resposta_codigo",
        "status_resposta",
        "resposta_valida"
    )
    .count()
    .orderBy("resposta_codigo")
)

In [0]:
codigos_validos = [
    1,
    2,
    97,
    98,
    99
]

codigos_invalidos = (
    df_cetic
    .filter(
        ~F.col("resposta_codigo")
        .isin(codigos_validos)
    )
    .count()
)

print(
    "Códigos de resposta fora do domínio:",
    codigos_invalidos
)

In [0]:
display(
    df_cetic.agg(

        F.min("peso").alias(
            "peso_minimo"
        ),

        F.max("peso").alias(
            "peso_maximo"
        ),

        F.avg("peso").alias(
            "peso_medio"
        ),

        F.sum(
            F.when(
                F.col("peso") <= 0,
                1
            ).otherwise(0)
        ).alias(
            "pesos_nao_positivos"
        )
    )
)

In [0]:
display(
    df_youtube.agg(

        F.count("*").alias(
            "total_registros"
        ),

        F.sum(
            F.when(
                F.col("video_id").isNull(),
                1
            ).otherwise(0)
        ).alias("video_id_nulo"),

        F.sum(
            F.when(
                F.col("data_trending").isNull(),
                1
            ).otherwise(0)
        ).alias("data_nula"),

        F.sum(
            F.when(
                F.col("categoria_original").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "categoria_original_nula"
        ),

        F.sum(
            F.when(
                F.col("views").isNull(),
                1
            ).otherwise(0)
        ).alias("views_nulo"),

        F.sum(
            F.when(
                F.col("likes").isNull(),
                1
            ).otherwise(0)
        ).alias("likes_nulo"),

        F.sum(
            F.when(
                F.col("comments").isNull(),
                1
            ).otherwise(0)
        ).alias("comments_nulo")
    )
)

In [0]:
duplicados_youtube = (
    df_youtube
    .groupBy(
        "video_id",
        "data_trending"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

print(
    "Duplicidades video_id + data_trending:",
    duplicados_youtube.count()
)

In [0]:
display(
    df_youtube.agg(
        F.min("data_trending").alias(
            "data_minima"
        ),
        F.max("data_trending").alias(
            "data_maxima"
        ),
        F.countDistinct(
            "video_id"
        ).alias(
            "videos_unicos"
        )
    )
)

In [0]:
fora_2025 = (
    df_youtube
    .filter(
        F.year("data_trending") != 2025
    )
    .count()
)

print(
    "Registros fora de 2025:",
    fora_2025
)

In [0]:
display(
    df_youtube.agg(

        F.sum(
            F.when(
                F.col("views") < 0,
                1
            ).otherwise(0)
        ).alias(
            "views_negativas"
        ),

        F.sum(
            F.when(
                F.col("likes") < 0,
                1
            ).otherwise(0)
        ).alias(
            "likes_negativos"
        ),

        F.sum(
            F.when(
                F.col("comments") < 0,
                1
            ).otherwise(0)
        ).alias(
            "comments_negativos"
        )
    )
)

In [0]:
display(
    df_youtube
    .groupBy(
        "nivel_comparabilidade"
    )
    .agg(
        F.count("*").alias(
            "registros"
        ),
        F.countDistinct(
            "video_id"
        ).alias(
            "videos_unicos"
        )
    )
    .orderBy(
        F.desc("registros")
    )
)

In [0]:
niveis_validos = [
    "Direta",
    "Aproximada",
    "Fraca",
    "Não comparável"
]

niveis_invalidos = (
    df_youtube
    .filter(
        ~F.col("nivel_comparabilidade")
        .isin(niveis_validos)
    )
    .count()
)

print(
    "Níveis inválidos:",
    niveis_invalidos
)

In [0]:
datas_inconsistentes = (
    df_youtube
    .filter(
        (F.col("data_trending")
         < F.col("primeira_data_trending"))
        |
        (F.col("data_trending")
         > F.col("ultima_data_trending"))
    )
    .count()
)

print(
    "Datas inconsistentes:",
    datas_inconsistentes
)

In [0]:
def analisar_outliers_iqr(
    df,
    coluna
):

    q1, q3 = df.approxQuantile(
        coluna,
        [0.25, 0.75],
        0.01
    )

    iqr = q3 - q1

    limite_superior = (
        q3 + 1.5 * iqr
    )

    quantidade = (
        df
        .filter(
            F.col(coluna)
            > limite_superior
        )
        .count()
    )

    return {
        "campo": coluna,
        "q1": q1,
        "q3": q3,
        "limite_superior": limite_superior,
        "possiveis_outliers": quantidade
    }

In [0]:
outliers_views = analisar_outliers_iqr(
    df_youtube,
    "views"
)

outliers_likes = analisar_outliers_iqr(
    df_youtube,
    "likes"
)

outliers_comments = analisar_outliers_iqr(
    df_youtube,
    "comments"
)

print(outliers_views)
print(outliers_likes)
print(outliers_comments)

In [0]:
display(
    df_trends.agg(

        F.count("*").alias(
            "total_registros"
        ),

        F.sum(
            F.when(
                F.col("data_semana").isNull(),
                1
            ).otherwise(0)
        ).alias("data_nula"),

        F.sum(
            F.when(
                F.col("categoria").isNull(),
                1
            ).otherwise(0)
        ).alias("categoria_nula"),

        F.sum(
            F.when(
                F.col("indice_trends").isNull(),
                1
            ).otherwise(0)
        ).alias("indice_nulo")
    )
)

In [0]:
display(
    df_trends.agg(

        F.min(
            "indice_trends"
        ).alias(
            "indice_minimo"
        ),

        F.max(
            "indice_trends"
        ).alias(
            "indice_maximo"
        ),

        F.sum(
            F.when(
                (
                    F.col("indice_trends") < 0
                )
                |
                (
                    F.col("indice_trends") > 100
                ),
                1
            ).otherwise(0)
        ).alias(
            "indices_fora_dominio"
        )
    )
)

In [0]:
duplicados_trends = (
    df_trends
    .groupBy(
        "data_semana",
        "categoria"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

print(
    "Duplicidades data_semana + categoria:",
    duplicados_trends.count()
)

In [0]:
categorias_trends_validas = [
    "Notícias",
    "Esportes",
    "Música",
    "Humor",
    "Games"
]

display(
    df_trends
    .groupBy("categoria")
    .count()
    .orderBy("categoria")
)

In [0]:
categorias_invalidas_trends = (
    df_trends
    .filter(
        ~F.col("categoria")
        .isin(categorias_trends_validas)
    )
    .count()
)

print(
    "Categorias inválidas:",
    categorias_invalidas_trends
)

In [0]:
display(
    df_trends
    .filter(
        F.col("data_semana")
        < F.lit("2025-01-01")
    )
    .groupBy(
        "data_semana"
    )
    .count()
    .orderBy(
        "data_semana"
    )
)

In [0]:
resultados = []

def registrar_resultado(
    fonte,
    dimensao,
    regra,
    observado,
    esperado,
    status,
    observacao=""
):
    resultados.append(
        (
            fonte,
            dimensao,
            regra,
            str(observado),
            str(esperado),
            status,
            observacao
        )
    )

In [0]:
registrar_resultado(
    "CETIC",
    "Integridade",
    "Total de registros",
    df_cetic.count(),
    196280,
    "OK"
    if df_cetic.count() == 196280
    else "REVISAR"
)

registrar_resultado(
    "CETIC",
    "Unicidade",
    "id_respondente + categoria",
    duplicados_cetic.count(),
    0,
    "OK"
    if duplicados_cetic.count() == 0
    else "REVISAR"
)

registrar_resultado(
    "CETIC",
    "Domínio",
    "Categorias fora do domínio",
    categorias_invalidas_cetic,
    0,
    "OK"
    if categorias_invalidas_cetic == 0
    else "REVISAR"
)

In [0]:
registrar_resultado(
    "YouTube",
    "Unicidade",
    "video_id + data_trending",
    duplicados_youtube.count(),
    0,
    "OK"
    if duplicados_youtube.count() == 0
    else "REVISAR"
)

registrar_resultado(
    "YouTube",
    "Validade",
    "Registros fora de 2025",
    fora_2025,
    0,
    "OK"
    if fora_2025 == 0
    else "REVISAR"
)

registrar_resultado(
    "YouTube",
    "Consistência",
    "Datas inconsistentes",
    datas_inconsistentes,
    0,
    "OK"
    if datas_inconsistentes == 0
    else "REVISAR"
)

In [0]:
registrar_resultado(
    "Google Trends",
    "Unicidade",
    "data_semana + categoria",
    duplicados_trends.count(),
    0,
    "OK"
    if duplicados_trends.count() == 0
    else "REVISAR"
)

registrar_resultado(
    "Google Trends",
    "Domínio",
    "Categorias fora do domínio",
    categorias_invalidas_trends,
    0,
    "OK"
    if categorias_invalidas_trends == 0
    else "REVISAR"
)

In [0]:
df_resultados_qualidade = (
    spark.createDataFrame(
        resultados,
        [
            "fonte",
            "dimensao",
            "regra",
            "valor_observado",
            "valor_esperado",
            "status",
            "observacao"
        ]
    )
    .withColumn(
        "data_execucao",
        F.current_timestamp()
    )
)

display(
    df_resultados_qualidade
)

In [0]:
(
    df_resultados_qualidade.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_quality.resultados_qualidade"
    )
)

In [0]:
df_resumo_qualidade = (
    df_resultados_qualidade
    .groupBy(
        "fonte",
        "status"
    )
    .count()
    .orderBy(
        "fonte",
        "status"
    )
)

display(
    df_resumo_qualidade
)

In [0]:
(
    df_resumo_qualidade.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.mvp_quality.resumo_qualidade"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_quality;

In [0]:
regras_revisar = (
    df_resultados_qualidade
    .filter(
        F.col("status") == "REVISAR"
    )
    .count()
)

print(
    "Regras que exigem revisão:",
    regras_revisar
)

if regras_revisar == 0:
    print(
        "Camada Silver aprovada para construção da Gold."
    )
else:
    print(
        "Existem regras que devem ser analisadas antes da Gold."
    )

## Conclusão da análise de qualidade

Foram avaliadas as três fontes utilizadas no MVP quanto às dimensões
de completude, unicidade, validade, consistência, domínio e integridade.

### CETIC

A granularidade foi validada pela combinação:

`id_respondente + categoria`

Os 24.535 respondentes e oito categorias resultam em 196.280
registros analíticos.

### YouTube

A repetição de um vídeo em diferentes datas foi considerada
comportamento esperado da fonte.

A granularidade adotada para validação é:

`video_id + data_trending`

Valores extremos de visualizações, curtidas e comentários foram
tratados como possíveis outliers analíticos e não foram removidos,
pois podem representar conteúdos genuinamente virais.

### Google Trends

Os índices foram validados dentro do domínio de 0 a 100.

A granularidade adotada é:

`data_semana + categoria`

A semana iniciada em 29/12/2024 foi preservada por representar
a semana de transição que contém o início de 2025.

Os resultados desta etapa são persistidos no schema `mvp_quality`
e utilizados como evidência de que as tabelas Silver estão aptas
à construção da camada Gold.